# 02 – Preprocessing (czyszczenie i inżynieria cech)
**Cel:** Usunięcie wartości odstających, obsługa braków, kodowanie zmiennych kategorycznych, inżynieria cech, zapis przetworzonego datasetu do `data/processed/`.

## 1. Import i wczytanie danych

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
from pathlib import Path
import os
PROJECT_ROOT = Path(os.getcwd())
# Jeśli CWD to notebooks/, cofnij się poziom wyżej
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH    = str(PROJECT_ROOT / 'data' / 'raw' / 'Car_Prices_Poland_Kaggle.csv')
PROC_PATH    = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
FIGURES_PATH = str(PROJECT_ROOT / 'reports' / 'figures') + os.sep
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIGURES_PATH:', FIGURES_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
print(f'Wczytano: {df.shape[0]} wierszy, {df.shape[1]} kolumn')
df.head()

## 2. Czyszczenie danych

In [ ]:
key_cols = ['year', 'mileage', 'vol_engine', 'fuel', 'mark', 'price']
df_clean = df.dropna(subset=key_cols).copy()
print(f'Po usunięciu braków: {df_clean.shape[0]} wierszy')

In [ ]:
print('Przed filtrowaniem:')
print(df_clean[['year','mileage','vol_engine','price']].describe())
df_clean = df_clean[
    (df_clean['price'] > 500) & (df_clean['price'] < 1_500_000) &
    (df_clean['mileage'] >= 0) & (df_clean['mileage'] < 1_500_000) &
    (df_clean['year'] >= 1990) & (df_clean['year'] <= 2022) &
    (df_clean['vol_engine'] >= 0) & (df_clean['vol_engine'] < 10_000)
]
print(f'\nPo filtrowaniu: {df_clean.shape[0]} wierszy')

In [ ]:
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Usunięto duplikatów: {n_before - len(df_clean)}')
print(f'Finalny rozmiar: {df_clean.shape}')

## 3. Normalizacja zmiennych kategorycznych

In [ ]:
for col in ['mark', 'model', 'fuel', 'city', 'province', 'generation_name']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.strip().str.lower()
print('Rodzaje paliwa po normalizacji:')
print(df_clean['fuel'].value_counts())

## 4. Inżynieria cech

In [ ]:
CURRENT_YEAR = 2022
df_clean['car_age'] = CURRENT_YEAR - df_clean['year']
df_clean['mileage_per_year'] = df_clean['mileage'] / (df_clean['car_age'].clip(lower=1))
df_clean['log_price'] = np.log1p(df_clean['price'])
df_clean['is_electric_hybrid'] = df_clean['fuel'].isin(['electric', 'hybrid']).astype(int)
print(df_clean[['car_age', 'mileage_per_year', 'log_price', 'is_electric_hybrid']].describe())

## 5. Kodowanie zmiennych kategorycznych

In [ ]:
mark_median = df_clean.groupby('mark')['price'].median()
df_clean['mark_median_price'] = df_clean['mark'].map(mark_median)
fuel_dummies = pd.get_dummies(df_clean['fuel'], prefix='fuel', drop_first=True)
df_clean = pd.concat([df_clean, fuel_dummies], axis=1)
print(f'Kolumny po kodowaniu: {df_clean.shape[1]}')
df_clean.head(3)

## 6. Wizualizacja po czyszczeniu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].hist(df_clean['price'], bins=60, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Cena (po czyszczeniu)')
axes[0].set_xlabel('PLN')
axes[1].hist(df_clean['log_price'], bins=60, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('log(Cena + 1)')
axes[1].set_xlabel('log PLN')
axes[2].hist(df_clean['car_age'], bins=30, color='seagreen', edgecolor='black', alpha=0.8)
axes[2].set_title('Wiek pojazdu (lata)')
axes[2].set_xlabel('Lata')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '02_clean_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Zapis przetworzonego datasetu

In [ ]:
df_clean.to_csv(PROC_PATH + 'cars_clean.csv', index=False)
print(f'Zapisano: {PROC_PATH}cars_clean.csv  ({df_clean.shape[0]} wierszy, {df_clean.shape[1]} kolumn)')